# Reachy 1.2 — Driving the Arm in the FWD Center Lab

A hands-on tutorial for the **FWDCenterLabMCC** scene: the measured MCC tabletop,
the 3×3 taped grid, and the 80/20 rig frame Reachy is bolted into.

Two routines, written to be read and modified:

1. **Place the arm on the table** — out of the rail pocket and onto the board,
   the motion Siva performs by hand (photos in
   `IITG-Reachy-Project/docs/pics/`).
2. **Raise the arm and move the gripper every way it moves** — one joint at a
   time, then in combination.

Each section says *what* the command does and *why* the numbers are what they
are, then runs it. Change the numbers and re-run — that is the point.

---

### Before you start

Launch the simulator on the lab scene so the physics and the RViz view both show
the real geometry:

```bash
REACHY_SIM_SCENE=FWDCenterLabMCC ./scripts/start_sim.sh
```

Watch **RViz at http://localhost:6080** and the **cameras at
http://localhost:8080** while cells run.

> **This is the simulator.** On the physical robot every motion must go through
> `src/reachy_ai/motion/primitives.py` with `REACHY_ENABLE_MOTION=true` and a
> human operator present. The angles below are a *teaching* device — once a
> routine settles, promote it to a named pose in `primitives.py`.

## 0. The rail pocket, and why the arm has to reverse out of it

Reachy's torso is bolted to a horizontal 80/20 frame. The right arm hangs down
through an opening in that frame — call it the **pocket**. Measured in the scene:

| | |
|---|---|
| right shoulder | world **(0.000, −0.190, 1.000)** |
| elbow, arm hanging at rest | world **(0.000, −0.190, 0.720)** |
| board, 1 in thick | surface z = **0.740**, underside z = **0.7146** |
| rig rail tops | z = **0.7146** — the board rests **on top of** them |
| board's robot-side edge | x = **0.160** *(still unmeasured)* |
| the pocket | x ∈ [−0.368, **0.160**], y ∈ [−0.304, −0.076] |

The pocket is **9 in wide** (y) and roughly 20 in long (x). The elbow sits
**20 mm below** the rail plane — it is *inside* the slot, not above it.

There is **no cross rail in front of the board**: that edge is finished with a
wooden trim strip overhanging open air (`docs/pics/80903693586`), so the pocket's
forward bound is the board itself, not aluminium.

That geometry dictates the escape:

- **Sideways is the short axis.** Swinging the arm out laterally — *abduction*,
  `r_shoulder_roll` — has only 114 mm before the outer rail. The elbow rises
  just 20 mm in the first 22° of roll, so it hits the rail before it clears it.
  Below −17.5° of roll the hanging arm is already in collision at rest.
- **Backwards is the long axis.** Swinging the arm back along the rail —
  *extension*, `r_shoulder_pitch` **positive**, with roll held at **0** — travels
  the long dimension. At +40° the elbow is at (−0.180, −0.190, **0.786**): it has
  moved 180 mm back, risen 66 mm, and **y has not changed at all**. It is now
  above the rail plane and out of the pocket.

Extension is capped around **+45°** with the arm straight — beyond that the
forearm reaches the rig's *back* rail. +40° is used here because it leaves the
most margin overall: at +45° the forearm comes within 1.7 mm of that back rail,
at +40° it has 44 mm.

Once the arm is clear, curling the forearm tight lifts the whole lower arm high
above the rails, and only then can the shoulder swing the folded unit forward and
out over the front rail onto the board.

> Checked exhaustively: with `r_shoulder_roll` pinned at 0 the arm can reach
> 28 044 configurations but **none** of them puts the hand over the table — the
> elbow crosses the board's near edge at z = 0.770 and the upper-arm capsule is
> 35 mm in radius, so it clips by 5 mm. The roll is needed on the way *in*, not
> on the way *out*. Every waypoint below was verified at 0.2° resolution against
> the board, all five rig rails, the pedestal and the robot itself.

## 1. Connect and take stock

In [1]:
import time
import sys
import pathlib

# Make src/ importable whether this runs in the container or from the repo.
for _c in (pathlib.Path("/opt/src"), pathlib.Path.cwd().parent / "src"):
    if _c.is_dir() and str(_c) not in sys.path:
        sys.path.insert(0, str(_c))

from reachy_sdk import ReachySDK                 # Reachy 1.2 SDK — NOT reachy2_sdk
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode
from reachy_ai.motion.safety import gate_check
from reachy_ai.scene.awareness import SceneModel

REACHY_HOST = "localhost"
REACHY_PORT = 50051        # fake_reachy_server.py; the physical robot uses 50055

reachy = ReachySDK(host=REACHY_HOST, sdk_port=REACHY_PORT)
arm = reachy.r_arm
print(f"connected to {REACHY_HOST}:{REACHY_PORT}")
print(f"right arm joints: {', '.join(arm.joints.keys())}")
print(f"safety gate_check(): {gate_check()}")

connected to localhost:50051
right arm joints: r_shoulder_pitch, r_shoulder_roll, r_arm_yaw, r_elbow_pitch, r_forearm_yaw, r_wrist_pitch, r_wrist_roll, r_gripper
safety gate_check(): True


`gate_check()` returns `True` here because `REACHY_SIM_BACKEND` is a simulation
backend. On hardware it returns `False` until `REACHY_ENABLE_MOTION=true` is set
with an operator present. **Never bypass it.**

### The scene, read from the same YAML the simulator loaded

In [2]:
SCENE_YAML = next(p for p in (
    pathlib.Path("/opt/scenes/FWDCenterLabMCC.yaml"),
    pathlib.Path.cwd().parent / "scenes" / "FWDCenterLabMCC.yaml",
) if p.exists())

scene = SceneModel.from_yaml(str(SCENE_YAML))
table = scene.table

print(f"scene file      : {SCENE_YAML}")
print(f"table surface z : {scene.table_surface_z:.3f} m")
print(f"table extent    : x {table.center[0]-table.size[0]/2:.3f} .. "
      f"{table.center[0]+table.size[0]/2:.3f}   "
      f"y {table.center[1]-table.size[1]/2:.3f} .. {table.center[1]+table.size[1]/2:.3f}")
print(f"rig rails       : {len([o for o in scene.static_obstacles() if 'rig-frame' in o.tags])}")
print()
print("addressable grid cells (robot's view: row 1 = nearest, col 1 = its left):")
for cid in scene.grid_cells():
    x, y, z = scene.cell_center(cid)
    print(f"   {cid}   x={x:+.4f}  y={y:+.4f}  z={z:.4f}")

scene file      : /opt/scenes/FWDCenterLabMCC.yaml
table surface z : 0.740 m
table extent    : x 0.160 .. 0.747   y -0.349 .. 0.349
rig rails       : 5

addressable grid cells (robot's view: row 1 = nearest, col 1 = its left):
   cell_r1c1   x=+0.2794  y=+0.1524  z=0.7400
   cell_r1c2   x=+0.2794  y=+0.0000  z=0.7400
   cell_r1c3   x=+0.2794  y=-0.1524  z=0.7400
   cell_r2c1   x=+0.4318  y=+0.1524  z=0.7400
   cell_r2c2   x=+0.4318  y=+0.0000  z=0.7400
   cell_r2c3   x=+0.4318  y=-0.1524  z=0.7400
   cell_r3c1   x=+0.5842  y=+0.1524  z=0.7400
   cell_r3c2   x=+0.5842  y=+0.0000  z=0.7400
   cell_r3c3   x=+0.5842  y=-0.1524  z=0.7400


Two of those nine — **`cell_r3c1` and `cell_r3c2`** — are out of the right arm's
reach (0.709 m and 0.649 m from the shoulder against a 0.609 m maximum). Siva
confirmed that on the physical robot. Plan tasks around the other seven.

## 2. The joint map and its sign conventions

Eight joints, all commanded in **degrees**. The signs are not all intuitive, so
keep this open while you experiment:

| joint | range | what a **positive** value does |
|---|---|---|
| `r_shoulder_pitch` | −150 … +90 | **extension** — swings the arm *backward* along the rail. Negative is **flexion**, forward. |
| `r_shoulder_roll`  | −180 … +10 | negative is **abduction** — swings the arm out sideways, across the rails |
| `r_arm_yaw`        | −90 … +90  | twists the upper arm about its own axis |
| `r_elbow_pitch`    | −125 … 0   | **only negative** — bends the elbow; 0 is a straight arm |
| `r_forearm_yaw`    | −100 … +100| rotates the forearm (pronate / supinate) |
| `r_wrist_pitch`    | −45 … +45  | tilts the hand up / down relative to the forearm |
| `r_wrist_roll`     | −45 … +45  | rolls the hand about the forearm axis |
| `r_gripper`        | −69 … +20  | **inverted: negative OPENS, positive CLOSES** |

Pitch and roll are the two that matter for getting out of the pocket, and they
are *not* interchangeable: pitch moves along the pocket's 19 in axis, roll across
its 9 in axis.

### The gripper sign is backwards from intuition

Measured from `reachy_1_2.xml`, the **right** gripper's pad gap:

| `r_gripper` | pad gap |
|---|---|
| −69° | 7.4 cm (fully open) |
| −45° | 6.5 cm |
| 0°   | 2.6 cm |
| +20° | 0.7 cm (closed) |

The **left** gripper's range is mirrored, so its signs are the opposite way
round. `primitives.open_gripper()` / `close_gripper()` take a `side=` argument
and handle that for you.

In [3]:
OPEN = -45.0   # gripper open   (~6.5 cm pad gap)
SHUT =  20.0   # gripper closed (~0.7 cm)

R_JOINTS = ["r_shoulder_pitch", "r_shoulder_roll", "r_arm_yaw", "r_elbow_pitch",
            "r_forearm_yaw", "r_wrist_pitch", "r_wrist_roll", "r_gripper"]
ARM7 = R_JOINTS[:7]

from reachy_ai.motion.kinematics import (CartesianPlanner, R_ARM_JOINTS,
                                         UnreachableError, link_capsules,
                                         link_frames)
from reachy_ai.motion.gaze import neck_angles_for, can_look_at

planner = CartesianPlanner(arm, scene, side="right")


def arm_q():
    """The seven arm angles, in the order the kinematics module expects."""
    return [getattr(arm, j).present_position for j in R_ARM_JOINTS]


def gripper_world_xyz():
    """Gripper pad position in world coordinates, via the SDK's own FK."""
    return planner.fk_world(arm_q())


def elbow_world_xyz():
    """Elbow position in world coordinates.

    Worth having next to the pad, because the two disagree about how safe a
    pose is.  The SDK's FK reports one point on the robot — the wrist, which we
    offset to the pad — and a pose that holds the pad high can still be resting
    the elbow on the table.  See PRESENT in the next cell for what that cost.
    """
    return tuple(float(v) for v in link_frames(arm_q())[1])


def grip_now():
    """The gripper's actual aperture, which sets how wide the hand really is.

    Not cosmetic: the moving finger swings out as the hand opens, from 3.4 cm
    off the wrist axis when shut to 8.2 cm wide open.  A guard that assumes a
    shut hand while the notebook hovers with an open one under-models the
    gripper by 3.2 cm — and did, until it threw foam_block 0.81 m.
    """
    return arm.r_gripper.present_position


def clearance(joints=None, statics=True):
    """Closest approach between ANY arm link and anything it must not touch.

    Negative means the model says they overlap.  ``statics`` includes the rig
    rails as well as the four objects on the board (but not the tabletop — see
    SceneModel.obstacle_ids for why that one is opt-in).
    """
    return planner.clearance(arm_q() if joints is None else joints,
                             include_static=statics)


# ── Where the head is looking ────────────────────────────────────────────────
#
# Raising the arm to keep the elbow off the table moved the working pose out of
# the stereo cameras' default view: the hand now does its work off frame while
# the cameras watch an empty rig.  So the head follows.
NECK = ("neck_roll", "neck_pitch", "neck_yaw")


def head_on():
    """Stiffen the neck.  turn_on('head') alone does NOT do this.

    The Orbita neck accepts joint goals only once the joints are explicitly
    non-compliant; without this the goals are taken and quietly ignored.
    """
    reachy.turn_on("head")
    for j in NECK:
        getattr(reachy.head, j).compliant = False
    time.sleep(0.3)


def look_at(target, secs=1.0, label=""):
    """Point the stereo cameras at a world point.  See motion/gaze.py."""
    pitch, yaw = neck_angles_for(target)
    cmd = {reachy.head.neck_pitch: pitch, reachy.head.neck_yaw: yaw,
           reachy.head.neck_roll: 0.0}
    goto(cmd, duration=secs, interpolation_mode=InterpolationMode.MINIMUM_JERK)
    goto(cmd, duration=secs * 0.7, interpolation_mode=InterpolationMode.MINIMUM_JERK)
    if label:
        print(f"  gaze -> {label}  (neck pitch {pitch:+.1f}, yaw {yaw:+.1f})")
    return pitch, yaw


def watch_hand(secs=1.0, label=""):
    """Follow the gripper.  Aims at the WRIST, not the pad: the pad swings
    about the wrist as the hand rolls, and a gaze chasing it wobbles."""
    _, _, wrist, _ = link_frames(arm_q())
    return look_at(tuple(float(v) for v in wrist), secs, label)


def watch_table(secs=1.2, label=""):
    """Look at the middle of the taped grid."""
    t = scene.table
    return look_at((t.center[0], t.center[1], scene.table_surface_z), secs, label)


def show_pose(label=""):
    print(label)
    print("   " + "  ".join(f"{j[2:]}={getattr(arm, j).present_position:+6.1f}"
                            for j in R_JOINTS))


reachy.turn_on("r_arm")
time.sleep(0.5)
show_pose("arm powered on, current pose:")
print(f"   gripper pad at world {tuple(round(v, 3) for v in gripper_world_xyz())}")
print(f"   elbow       at world {tuple(round(v, 3) for v in elbow_world_xyz())}")
print(f"   nearest approach     {clearance()}")

arm powered on, current pose:
   shoulder_pitch=  +0.0  shoulder_roll=  +0.0  arm_yaw=  -0.0  elbow_pitch=  -0.0  forearm_yaw=  -0.0  wrist_pitch=  -0.0  wrist_roll= +40.1  gripper= -40.1
   gripper pad at world (0.0, -0.113, 0.378)
   elbow       at world (0.0, -0.19, 0.72)
   nearest approach     upper_arm clears rig_rail_inner_right by 7.9 cm


### How to actually command a move

Two things about the MuJoCo physics backend, both learned the hard way:

- **Setting `goal_position` once does nothing.** Under `mujoco-remote` the arm
  only tracks while setpoints are being *streamed*. Holding a goal for 30 s
  leaves the arm exactly where it started.
- **Hand-rolled 25 Hz interpolation tracks badly.** Ramping all joints in a
  Python loop leaves 20–60° of steady-state error — the elbow sags out of its
  fold entirely, which wrecks a route that depends on staying folded.

Use the SDK's own `goto()` with minimum-jerk instead. It tracks to **≈2°** on
every joint. Because minimum jerk applies one shared time profile to all joints,
the *path through joint space is still the straight line* between waypoints —
which is exactly what the collision verification below assumes.

`move_to` also **checks that each waypoint was actually reached** before moving
on, and raises `TrackingError` if not. That matters here: the corridor out of the
pocket is a few millimetres wide, so a waypoint reached 10° short is no longer
the pose that was verified, and continuing from it is how the arm ends up jammed
against a rail. A `TrackingError` is the routine refusing to crash the arm — give
that segment a longer duration and re-run.

**Two thresholds, deliberately separate.** `converge_tol` decides how hard
`move_to` tries — how close is close enough to stop re-streaming setpoints —
and defaults to `TRACK_TOL` always. `tol` decides whether falling short is
fatal, and that is the one to loosen when the arm is in open air.

They used to be a single parameter, and merging them was a bug with teeth: the
retry loop exited on the same test that triggered the raise, so passing the loose
`LESSON_TOL` to skip the *abort* also skipped the *retries*. A move got one
0.8 s settle instead of up to four, stopped wherever it had got to, and printed a
line that looked like success. In §4.7 that left the arm 20–42° from its solved
pose, hovering up to 23.6 cm from the cell it had just named — further than the
15.2 cm between cells.

If you loosen a tolerance, be clear which question you are answering: *"don't
abort"* is `tol`, *"don't bother arriving"* is `converge_tol`. They are almost
never the same wish.

In [4]:
class TrackingError(RuntimeError):
    """The physics arm did not reach a waypoint closely enough to continue."""


# The corridor out of the pocket is only a few millimetres wide, so a waypoint
# that is several degrees short is no longer the pose that was verified.  Rather
# than let that error compound into the next segment (which is how the arm ends
# up jammed against a rail), refuse to continue.
TRACK_TOL = 6.0     # degrees

# Which joints the guard actually polices.  shoulder pitch/roll, arm_yaw, elbow
# and wrist_pitch are what put the elbow, forearm and hand where the clearance
# was measured — those get TRACK_TOL.  r_wrist_roll, r_forearm_yaw and the
# gripper only spin the hand about its own axis; they are weak joints (kp=60,
# 10 Nm) that converge over several waypoints, and a few degrees of error on
# them moves the pad by millimetres inside margins of 20 mm or more.  Holding
# them to 6 deg aborts a route that is in no danger.
CRITICAL = ("r_shoulder_pitch", "r_shoulder_roll", "r_arm_yaw",
            "r_elbow_pitch", "r_wrist_pitch")
LOOSE_TOL = 90.0

# Routine 2 runs at PRESENT, high above the board with the rig far away.  The
# transit guard is there to stop the arm being driven into the rig; it has no
# job here, and aborting would hide the joint behaviour the lesson exists to
# show.  So the sweeps report their tracking error rather than enforcing it.
#
# LESSON_TOL SWITCHES OFF THE ABORT, NOT THE CONVERGENCE.  It used to do both,
# because one `tol` served the retry loop and the raise.  With tol=90 the loop's
# exit test passed on the first pass, so a "lesson" move got ONE 0.8 s settle
# instead of up to four progressively longer ones — and the arm simply stopped
# short.  Measured on the grid sweep in 4.7, that left it 20-42 deg from its
# solved pose and the pad up to 23.6 cm from the cell it had named, while
# printing a line that read like success.  The two thresholds are now separate:
# `converge_tol` decides how hard move_to tries, `tol` decides whether falling
# short is fatal.
LESSON_TOL = 90.0


def joint_error(target, critical_only=True):
    names = [n for n in target if n in CRITICAL] if critical_only else \
            [n for n in target if n != "r_gripper"]
    return max((abs(getattr(arm, n).present_position - target[n]), n) for n in names)


def loose_error(target):
    names = [n for n in target
             if n not in CRITICAL and n != "r_gripper"]
    if not names:
        return (0.0, "")
    return max((abs(getattr(arm, n).present_position - target[n]), n) for n in names)


def prime_arm(secs=0.6):
    """Send one throwaway command before the routine needs one to land.

    INSURANCE, NOT A PROVEN FIX — the honest state of this is worth writing down
    because the failure it targets is intermittent and its cause is not known.

    What is certain is the symptom.  Twice, on a connection made shortly after
    the simulator was restarted:

        GRIP_SHUT   (3.5s)  worst joint error  0.0 deg      <- passes
        TrackingError: BACK: r_shoulder_pitch is 40.0 deg from its goal

    40 deg from a +40 target is not tracking error, it is an arm that never
    moved at all.  GRIP_SHUT hides it because only the gripper changes there and
    `joint_error` does not police the gripper, so a dead command stream reports
    success and the failure surfaces one move later, looking like a guard fault.

    What is NOT established is the cause.  The obvious theory — the bridge drops
    the first goto of a fresh connection — does not survive testing: two
    controlled runs on a freshly restarted simulator both got through Routine 1,
    one WITH this call and one WITHOUT, so it cannot be credited with the pass.
    A single-joint goto does behave differently from the full-pose gotos this
    notebook sends, which may be part of it, but that is a lead and not a cause.

    So this stays: a no-op costing 0.6 s that cannot hurt and might help.  The
    part that reliably earns its place is the diagnostic in `move_to` — when the
    arm has not moved at all it now says so, instead of blaming the route.  If
    you hit it, re-run the cell; that has cleared it every time.
    """
    here = {getattr(arm, j): getattr(arm, j).present_position for j in R_JOINTS}
    goto(here, duration=secs, interpolation_mode=InterpolationMode.MINIMUM_JERK)


def clip_target(label, target, margin):
    """Shorten a move until no part of the arm comes within `margin` of anything.

    Returns the target to fly — the original if the whole move is clear, a
    shortened one if it is not, or None if there is nothing safe to fly at all.

    Three things make this a real check rather than the pad-height check it
    replaces, and each corresponds to a way that one was wrong:

    * it uses the WHOLE ARM (three capsules from the MJCF collision geoms), not
      the gripper pad.  At the pose this notebook used to sweep from, the pad
      was 21 cm above the board and the forearm was 5 mm inside red_cube;
    * it checks the PATH, not the endpoints.  `goto` interpolates in joint
      space, so the arm between two clear poses is not itself clear;
    * it starts from where the arm ACTUALLY IS, not from the pose it was last
      told to hold.  Under physics those differ by degrees, and it is the real
      links that have to miss the real cube.

    It refreshes the object positions first, because a guard is only as good as
    its idea of where things are, and objects move — the robot moves them on
    purpose, and it moved them by accident until this notebook was fixed.
    Checking against where an object was PLACED is the same mistake in a
    different costume: the model and the world disagree, and the model wins an
    argument it should lose.

    What it still cannot see is tracking error in the move it is about to make:
    the model knows where the links would be at the commanded pose, not where
    the physics will leave them.  That is what the margin is for, and it is why
    scene_drift() below remains the actual evidence.
    """
    refresh_scene()
    here = arm_q()
    want = dict(zip(R_ARM_JOINTS, here))
    want.update({j: v for j, v in target.items() if j in R_ARM_JOINTS})
    goal = [want[j] for j in R_ARM_JOINTS]
    clipped, frac, c = planner.clip(here, goal, margin=margin,
                                    gripper_deg=grip_now(),
                                    include_static=True)
    if frac > 0.999:
        return target
    if frac < 0.02:
        print(f"  {label:11s} REFUSED — {c}; no part of this move keeps "
              f"{margin * 100:.0f} cm of air")
        return None
    out = dict(target)
    out.update(dict(zip(R_ARM_JOINTS, clipped)))
    changed = ", ".join(f"{j[2:]} {target[j]:+.0f}->{out[j]:+.0f}"
                        for j in target if j in R_ARM_JOINTS
                        and abs(target[j] - out[j]) > 0.5)
    print(f"  {label:11s} CLIPPED to {frac * 100:.0f}% of the way "
          f"({changed}) — {c}")
    return out


def move_to(label, target, secs, report=True, settle=0.8, tol=TRACK_TOL,
            retries=3, converge_tol=None, clear=None):
    """Interpolate to `target` with the SDK's minimum-jerk trajectory generator.

    A short second pass to the same target re-streams the setpoints and pulls out
    the tracking lag; without it the physics arm can finish a fast segment
    10-20 deg short, and simply *holding* a goal will not close the gap — under
    `mujoco-remote` the arm only moves while setpoints are streaming.

    Two independent thresholds, and keeping them apart matters:

    * `converge_tol` — how close is close enough to STOP re-streaming.  Defaults
      to TRACK_TOL whatever `tol` is, so a move always tries just as hard to
      arrive.  Raise it only to deliberately sample a joint mid-flight.
    * `tol` — how far short is far enough to ABORT.  This is the safety guard,
      and it is the one to loosen in open air.

    `clear` is the third, and it guards something the other two cannot see.  A
    move can track perfectly and still be wrong, because tracking says nothing
    about what the arm passes through on the way.  Give it a margin in metres
    and the move is shortened to keep every link that far from every object on
    the board (and from the rig).  Routine 1 leaves it off: that route is
    verified waypoint by waypoint against the rig, and clipping it would abandon
    the corridor it was measured in.

    USE IT ON OUTBOUND MOVES ONLY.  A refusal means "do not move", and that is
    the wrong answer for a move whose purpose is to get OUT of a bad place.
    Observed: with the guard on the return to PRESENT, a run that had already
    disturbed the board refused to retreat — `PRESENT REFUSED, hand clears
    soda_can by 2.7 cm` — and left the arm parked over the wreckage it had just
    made, when the one thing it should have done was leave.  Retreats to a known
    pose go unguarded, deliberately.
    """
    if clear is not None:
        target = clip_target(label, target, clear)
        if target is None:
            return
    cmd = {getattr(arm, n): v for n, v in target.items()}
    ctol = TRACK_TOL if converge_tol is None else converge_tol
    # Where the arm started, so a failure can tell "tracked badly" from "never
    # moved".  They need different answers and they used to read identically.
    was = {n: getattr(arm, n).present_position for n in target if n in CRITICAL}
    goto(cmd, duration=secs, interpolation_mode=InterpolationMode.MINIMUM_JERK)
    # Progressively longer settles.  The wrist joints are weak (kp=60,
    # forcerange 10) and a single short pass leaves a large offset half-closed.
    for k in range(retries + 1):
        if settle:
            goto(cmd, duration=settle * (1 + k),
                 interpolation_mode=InterpolationMode.MINIMUM_JERK)
        if joint_error(target)[0] <= ctol and loose_error(target)[0] <= LOOSE_TOL:
            break
    err, worst_joint = joint_error(target)
    lerr, ljoint = loose_error(target)
    if lerr > LOOSE_TOL:
        raise TrackingError(
            f"{label}: {ljoint} is {lerr:.1f} deg from its goal (loose tolerance "
            f"{LOOSE_TOL:.0f}).  Even the non-critical joints are not tracking."
        )
    if err > tol:
        moved = max((abs(getattr(arm, n).present_position - v)
                     for n, v in was.items()), default=0.0)
        if moved < 0.5:
            raise TrackingError(
                f"{label}: the arm did not move AT ALL ({worst_joint} is still "
                f"{err:.1f} deg from its goal).  This is not tracking error — "
                f"the simulator dropped the command stream, which it does to "
                f"the first goto after a fresh connection.  Call prime_arm() "
                f"before the routine, or just re-run this cell."
            )
        raise TrackingError(
            f"{label}: {worst_joint} is {err:.1f} deg from its goal (tolerance "
            f"{tol:.0f}).  The arm is not where the verified route assumes; "
            f"continuing would drive it into the rig.  Re-run the cell, or give "
            f"this segment a longer duration."
        )
    if report:
        x, y, z = gripper_world_xyz()
        print(f"  {label:11s} ({secs:.1f}s)  worst joint error {err:4.1f} deg "
              f"({worst_joint[2:]})   pad -> ({x:+.3f}, {y:+.3f}, {z:.3f})")


LIVE_POSES = "/tmp/reachy_scene_overrides.json"

# Where each object started, captured once.  scene_drift measures against THIS,
# not against the SceneModel — refresh_scene() moves the model to wherever the
# objects actually are, so a model-vs-live comparison would report zero drift by
# construction.  The guard needs to know where things are; the verification
# needs to know where they were.  Different questions, different reference
# points, and collapsing them would quietly disable the only honest check here.
SCENE_ORIGIN = {oid: scene.get(oid).center for oid in scene.manipulable_ids()}


def live_poses(path=LIVE_POSES):
    """The simulator's current object centres, or None if unavailable."""
    import json as _json
    try:
        return _json.load(open(path))
    except Exception:
        return None


def refresh_scene(path=LIVE_POSES):
    """Move the SceneModel's objects to where they actually are right now.

    Under mujoco-remote the container mirrors tracked object poses into `path`
    at 15 Hz.  Without that file the model keeps its as-loaded positions, which
    is the best it can do — and is why a run with no live feed should not be
    trusted to have guarded anything.
    """
    poses = live_poses(path)
    return scene.update_poses(poses) if poses else None


from reachy_ai.motion.escort import binding as clearance_binding
from reachy_ai.motion.escort import escort as _escort


def escort_to(label, target, secs, margin, legs=6, gripper_deg=None,
              report=True, margins=None, **clearance_kw):
    """`move_to` with the guard closed around it.

    `clip_target` and `move_to(clear=...)` check the move BEFORE it happens, and
    that is all they can do: they describe an arm that tracks perfectly, which
    this one does not.  The gap was measured — planning each grid cell, flying
    it, and recomputing clearance from the angles actually reached degraded the
    answer by at most 0.9 cm — and then `cell_r2c1` reported +5.5 cm of air to
    the soda can and moved it 0.189 m.  Both numbers are true.  The 0.9 cm was
    measured at the ENDPOINTS of each move; the can was hit in the MIDDLE of
    one.

    `joint_path` models a `goto` as a straight line in joint space, and each
    joint does run monotonically from start to goal — but not in step.  The
    shoulder (kp 300) arrives well before the wrists (kp 60), so the arm bows
    out of the corridor the guard cleared.  Both ends check out; the belly of
    the move does not, and no margin fixes that because nothing bounds the
    deviation before the move.

    So: fly it in `legs` short hops.  Before each hop, check it the old way —
    but FROM THE POSE THE ARM IS ACTUALLY IN, which is already a closed loop.
    After each hop, read the arm back and measure the clearance that really
    happened.  If that has eaten past `margin / 2`, stop and retreat to the last
    hop whose measurement was good.

    What it does NOT do is prevent the first contact: a hop is detected after it
    is flown.  The guarantee is that the REST of the move does not happen, which
    is the difference between one nudged can and a swept board.  `scene_drift`
    is still the evidence; this is what keeps one disturbance from becoming five.
    """
    want = dict(zip(R_ARM_JOINTS, arm_q()))
    want.update({j: v for j, v in target.items() if j in R_ARM_JOINTS})
    q_to = [want[j] for j in R_ARM_JOINTS]

    # The aperture the hand will be at IN TRANSIT, which is what sizes the hand
    # capsule.  An open hand is a 7.5 cm tube about the wrist axis against a
    # shut one's 5.2 cm; getting this wrong is what let the foam block move
    # 0.812 m while the guard reported 6.98 cm.
    grip = target.get("r_gripper", arm.r_gripper.present_position)
    if gripper_deg is None:
        gripper_deg = grip

    def send(q, dt):
        pose = dict(zip(R_ARM_JOINTS, q), r_gripper=grip)
        # tol=LESSON_TOL: a leg falling short is not fatal here, because the
        # next leg is planned from wherever it actually got to.  That is the
        # whole point — mis-tracking is data, not an abort condition.
        move_to(label, pose, dt, report=False, tol=LESSON_TOL,
                retries=1, settle=0.4)

    result = _escort(planner, q_to, send, arm_q, margin=margin,
                     margins=margins, legs=legs, duration=secs,
                     gripper_deg=gripper_deg, refresh=refresh_scene,
                     **clearance_kw)
    if report and not result.completed:
        print(f"  {label:11s} {result}")
    return result


# 5 mm, not 20.  At 20 mm a run reported "every object still on its cell"
# while red_cube had actually finished 13.1 mm off it — a check that says clean
# when something moved is the exact failure this notebook exists to stop making.
# 5 mm is above the physics' own resting jitter (measured < 1 mm) and below
# anything a person would call undisturbed.
DRIFT_TOL = 0.005


def scene_drift(tag="", path=LIVE_POSES, tol=DRIFT_TOL):
    """Compare LIVE object poses against where they started.

    The counterpart to clip_target(), and the reason both exist.  clip_target
    *predicts* — it is a geometric model of an arm that tracks perfectly, run
    before the move.  This *verifies*, by looking at the objects themselves
    afterwards.  A model can be wrong; the objects cannot.

    Under mujoco-remote the container mirrors tracked object poses into `path`
    at 15 Hz; without that file there is nothing to compare, and we say so
    rather than reporting a clean bill of health.
    """
    live = live_poses(path)
    if live is None:
        print(f"  drift[{tag}]: UNAVAILABLE — cannot confirm the scene is "
              f"undisturbed")
        return None
    moved = []
    for oid in scene.manipulable_ids():
        if oid not in live:
            continue
        want, got = SCENE_ORIGIN[oid], live[oid]
        d = sum((a - b) ** 2 for a, b in zip(want, got)) ** 0.5
        if d > tol:
            moved.append((oid, d))
    if moved:
        print(f"  drift[{tag}]: " + ", ".join(f"{o} moved {d:.3f} m"
                                              for o, d in moved))
    else:
        print(f"  drift[{tag}]: every object still on its cell")
    return moved

## 3. Routine 1 — out of the pocket and onto the board

The motion in four moves, exactly as it is done by hand on the robot:

1. **Back out of the pocket.** `r_shoulder_pitch` → **+40**, roll held at **0**.
   Pure extension along the rail. The elbow travels 180 mm backward and 66 mm up;
   *y never changes*.
2. **Curl the forearm up tight.** Elbow to **−125**, wrist tucked, then extend a
   little further to +70 so the whole folded lower arm rides ~164 mm above the
   rail plane.
3. **Swing the folded unit forward and out over the rail.** Pitch sweeps
   +70 → −17.5 while the roll opens to −37.5, carrying the compact arm over the
   front rail and the table's near edge.
4. **Unfold onto the board.** Elbow opens −120 → −45 and the forearm settles.

| # | pose | pitch / roll | elbow | wrist P/R | grip | elbow world |
|---|---|---|---|---|---|---|
| 0 | `HOME` | 0 / 0 | 0 | 0 / 0 | open | (0.000, −0.190, **0.720**) |
| 1 | `GRIP_SHUT` | 0 / 0 | 0 | 0 / 0 | **shut** | (0.000, −0.190, 0.720) |
| 2 | `BACK` | **+40** / 0 | 0 | 0 / 0 | shut | (−0.180, −0.190, **0.786**) |
| 3 | `CURL` | +40 / 0 | **−125** | +45 / 0 | shut | (−0.180, −0.190, 0.786) |
| 4 | `CURL_HIGH` | **+70** / 0 | −120 | +45 / 0 | shut | (−0.263, −0.190, **0.904**) |
| 5 | `TUCK` | +70 / 0 | −120 | **−45** / 0 | shut | (−0.263, −0.190, 0.904) |
| 6 | `SWING_1` | +37.5 / **−32.5** | −120 | −45 / 0 | shut | (−0.144, −0.340, 0.813) |
| 7 | `SWING_2` | +20 / −35 | −120 | −45 / 0 | shut | (−0.078, −0.351, 0.784) |
| 8 | `SWING_3` | **−17.5** / −37.5 | −120 | −45 / 0 | shut | (+0.067, −0.360, 0.788) |
| 9 | `HOVER` | −40 / −10 | **−60** | −15 / 0 | shut | (+0.177, −0.239, 0.789) |
| 10 | `REST_SHUT` | −40 / −10 | **−45** | −10 / **+30** | shut | (+0.177, −0.239, 0.789) |
| 11 | `REST` | −40 / −10 | −45 | −10 / +30 | **open** | (+0.177, −0.239, 0.789) |

Read the elbow column top to bottom. Rows 0→5 hold **y = −0.190 exactly** — the whole escape happens without a single degree of lateral motion. Only at
row 6, once the elbow is well clear of the rails, does y start to move.

The gripper closes for the transit: an open finger sweeps a wider volume and
catches the front rail. It opens again once the arm is down.

Every segment is contact-free except the last two, which register **−0.97 mm**
against the board. That is the forearm coming to rest — the goal, not a fault.

Measured clearance, worst point in each segment (true surface-to-surface gap):

| segment | gap | closest pair |
|---|---|---|
| HOME → GRIP_SHUT | +79.4 mm | upper arm ↔ inner-right rail |
| GRIP_SHUT → BACK | +44.2 mm | forearm ↔ back rail |
| BACK → CURL | +26.8 mm | thumb pad ↔ board |
| CURL_HIGH → TUCK | +83.9 mm | finger pad ↔ outer-right rail |
| TUCK → SWING_1 | +26.8 mm | forearm ↔ outer-right rail |
| SWING_1 → SWING_2 | +23.4 mm | forearm ↔ board |
| SWING_2 → SWING_3 | +25.9 mm | upper arm ↔ outer-right rail |
| SWING_3 → HOVER | **+4.8 mm** | upper arm ↔ board |

The 4.8 mm is the elbow crossing the board's robot-side edge, and it is the
tightest point on the route. Everything else has centimetres. See §6.

In [5]:
# ── Verified pose set for FWDCenterLabMCC ────────────────────────────────────
# Checked at 0.2 deg resolution against the board, all five rig rails, the
# pedestal and the robot's own links.  Do not tweak blind — the corridor out of
# the pocket is only a few millimetres wide in places.

def pose(**kw):
    base = dict.fromkeys(ARM7, 0.0)
    base["r_gripper"] = OPEN
    base.update(kw)
    return base


HOME      = pose()
GRIP_SHUT = pose(r_gripper=SHUT)

# 1. back out of the pocket — extension only, roll stays at 0
BACK      = pose(r_gripper=SHUT, r_shoulder_pitch=40.0)

# 2. curl the forearm up tight against the upper arm
CURL      = pose(r_gripper=SHUT, r_shoulder_pitch=40.0,
                 r_elbow_pitch=-125.0, r_wrist_pitch=45.0)
CURL_HIGH = pose(r_gripper=SHUT, r_shoulder_pitch=70.0,
                 r_elbow_pitch=-120.0, r_wrist_pitch=45.0)
TUCK      = pose(r_gripper=SHUT, r_shoulder_pitch=70.0,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)

# 3. swing the folded unit forward and out over the rail
SWING_1   = pose(r_gripper=SHUT, r_shoulder_pitch=37.5, r_shoulder_roll=-32.5,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)
SWING_2   = pose(r_gripper=SHUT, r_shoulder_pitch=20.0, r_shoulder_roll=-35.0,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)
SWING_3   = pose(r_gripper=SHUT, r_shoulder_pitch=-17.5, r_shoulder_roll=-37.5,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)

# 4. unfold onto the board
HOVER     = pose(r_gripper=SHUT, r_shoulder_pitch=-40.0, r_shoulder_roll=-10.0,
                 r_elbow_pitch=-60.0, r_wrist_pitch=-15.0)
REST_SHUT = pose(r_gripper=SHUT, r_shoulder_pitch=-40.0, r_shoulder_roll=-10.0,
                 r_elbow_pitch=-45.0, r_wrist_pitch=-10.0, r_wrist_roll=30.0)
REST      = dict(REST_SHUT, r_gripper=OPEN)

# Raised pose for routine 2.  Not eyeballed: this is the pose that keeps every
# LINK of the arm clear of every object through every sweep in section 4.
#
# It used to be (-37.5, -2.0, -80.0), described as "collision-free, so joints
# can be swept to their limits".  That was measured on the gripper pad, which
# sat 21 cm above the board and looked entirely safe.  The pad is not the arm:
# at that pose the ELBOW was at z = 0.778, 3.8 cm above a 0.740 tabletop and
# directly over the near-right grid cell where red_cube stands, with the forearm
# already 5 mm INSIDE the cube before any joint moved.
#
# Lifting the hand does not fix that.  The elbow rides a fixed 0.28 m sphere
# about the shoulder; red_cube's nearest surface is 0.320 m from the shoulder
# and the upper arm's own surface reaches 0.315 m, so the near-right cell lies
# inside the elbow's arc no matter how the wrist is held.  The arm has to be
# pointed where the arc does not sweep: up, and out to the right.
#
#     worst clearance, whole arm, over the whole of routine 2:
#         (-37.5,  -2.0, -80.0)   -1.9 cm   already inside red_cube
#         (-70.0, -25.0, -80.0)  +10.9 cm   vs red_cube, via the upper arm
#                                + 8.9 cm   counting the table and rig as well
PRESENT   = pose(r_shoulder_pitch=-70.0, r_shoulder_roll=-25.0,
                 r_elbow_pitch=-80.0)

# (name, pose, seconds, tolerance).  Tolerance is per-waypoint because the
# waypoints are not equally dangerous: settling into GRIP_SHUT happens with
# 79 mm of clearance all round and only has to undo the wrist's gravity drift,
# whereas SWING_3 -> HOVER passes the elbow 4.8 mm from the board's edge.
PLACE_ROUTE = [("GRIP_SHUT", GRIP_SHUT, 3.5, 25.0),
               ("BACK",      BACK,      3.0, TRACK_TOL),
               ("CURL",      CURL,      3.0, TRACK_TOL),
               ("CURL_HIGH", CURL_HIGH, 2.0, TRACK_TOL),
               ("TUCK",      TUCK,      2.0, TRACK_TOL),
               ("SWING_1",   SWING_1,   3.0, TRACK_TOL),
               ("SWING_2",   SWING_2,   2.5, TRACK_TOL),
               ("SWING_3",   SWING_3,   2.5, TRACK_TOL),
               ("HOVER",     HOVER,     2.5, TRACK_TOL),
               ("REST_SHUT", REST_SHUT, 3.0, TRACK_TOL),
               ("REST",      REST,      3.0, TRACK_TOL)]

# The stow route is the placement route run backwards.  Nothing may cut across
# it: a direct move from anywhere over the board to HOME drives the upper arm
# through the board's near edge.
STOW_ROUTE = [("REST_SHUT", REST_SHUT, 2.0, TRACK_TOL),
              ("HOVER",     HOVER,     2.0, TRACK_TOL),
              ("SWING_3",   SWING_3,   2.5, TRACK_TOL),
              ("SWING_2",   SWING_2,   2.5, TRACK_TOL),
              ("SWING_1",   SWING_1,   2.0, TRACK_TOL),
              ("TUCK",      TUCK,      3.0, TRACK_TOL),
              ("CURL_HIGH", CURL_HIGH, 2.0, TRACK_TOL),
              ("CURL",      CURL,      2.0, TRACK_TOL),
              ("BACK",      BACK,      3.0, TRACK_TOL),
              ("GRIP_SHUT", GRIP_SHUT, 3.0, 25.0),
              ("HOME",      HOME,      2.5, 25.0)]


# The joints that decide where the arm sits in the rig.  The wrist angles and
# the gripper do not move the elbow or forearm through the rails, and on a
# freshly reset sim they read wherever gravity left them while the motors were
# off (wrist_roll drifts to ~40 deg, the gripper falls open) — so judging "is
# the arm home?" on those would report a clean reset as a fault.
GROSS_JOINTS = ["r_shoulder_pitch", "r_shoulder_roll", "r_arm_yaw", "r_elbow_pitch"]


def at_pose(target, tol=8.0, joints=None):
    names = joints if joints is not None else [n for n in target if n != "r_gripper"]
    return all(abs(getattr(arm, n).present_position - target[n]) <= tol for n in names)


def pose_distance(target):
    return max(abs(getattr(arm, n).present_position - v)
               for n, v in target.items() if n != "r_gripper")


def ensure_home():
    """Put the arm back in the pocket before starting a placement.

    Makes the placement cell safe to re-run, and safe to run after an aborted
    one.  Jumping straight to HOME from anywhere over the board would cut the
    corner through the rig's front rail, and so would jumping to the *start* of
    the stow route.  Instead: find the waypoint the arm is already closest to,
    ease onto it, and retrace the route from there.
    """
    if at_pose(HOME, joints=GROSS_JOINTS):
        print("arm already at HOME (wrist/gripper will be set by the first move)\n")
        return
    order = [n for n, _, _, _ in STOW_ROUTE]
    by_name = {n: t for n, t, _, _ in STOW_ROUTE}
    nearest = min(order, key=lambda n: pose_distance(by_name[n]))
    print(f"arm is not at HOME (nearest waypoint: {nearest}, "
          f"{pose_distance(by_name[nearest]):.1f} deg away) — retracing from there")
    try:
        # Ease onto the nearest waypoint slowly, with a loose tolerance: this is
        # the one move on an unverified path, so keep it small and gentle.
        move_to(nearest, by_name[nearest], 4.0, report=False, tol=12.0, retries=3)
        for name, target, secs, tol in STOW_ROUTE[order.index(nearest) + 1:]:
            move_to(name, target, secs, report=False, tol=tol)
    except TrackingError as exc:
        raise TrackingError(
            f"cannot recover to HOME: {exc}\n\n"
            "The arm is most likely WEDGED in the rig — an interrupted routine "
            "can leave the forearm threaded under the front rail, where no "
            "commanded pose will pull it back out (the pad ends up below the "
            "tabletop, z < 0.74).  Reset the simulator rather than fighting it:\n"
            "    REACHY_SIM_SCENE=FWDCenterLabMCC ./scripts/start_sim.sh\n"
            "then re-run this notebook from the top."
        ) from None
    print(f"   back at HOME (within 8 deg: {at_pose(HOME, joints=GROSS_JOINTS)})\n")


print(f"{len(PLACE_ROUTE)} placement waypoints, {len(STOW_ROUTE)} stow waypoints")

11 placement waypoints, 11 stow waypoints


### Running it — about 35 s. Watch RViz.

In [7]:
reachy.turn_on("r_arm")
time.sleep(0.5)
prime_arm()          # the bridge drops its first command; spend it on a no-op
ensure_home()

print("out of the pocket and onto the board — watch RViz at localhost:6080\n")
for name, target, secs, tol in PLACE_ROUTE:
    move_to(name, target, secs, tol=tol)
    time.sleep(0.3)

scene_drift("after routine 1")
print("\narm resting on the board.")
show_pose("final pose:")

arm already at HOME (wrist/gripper will be set by the first move)

out of the pocket and onto the board — watch RViz at localhost:6080

  GRIP_SHUT   (3.5s)  worst joint error  0.0 deg (shoulder_roll)   pad -> (+0.000, -0.190, 0.350)


TrackingError: BACK: r_shoulder_pitch is 40.0 deg from its goal (tolerance 6).  The arm is not where the verified route assumes; continuing would drive it into the rig.  Re-run the cell, or give this segment a longer duration.

### What to try

- **Try the sideways version and watch it fail.** Replace `BACK` with
  `pose(r_gripper=SHUT, r_shoulder_roll=-45.0)` — abduction instead of
  extension. Under the MuJoCo backend the arm jams against the outer rail. That
  jamming *is* the feedback; leave the physics backend on rather than switching
  to `kinematic` to make it look clean.
- **Under-curl the elbow.** Set `CURL_HIGH`'s `r_elbow_pitch` to `-100` and the
  forearm drops back toward the rail plane; `SWING_1` then drives it into the
  outer-right rail.
- **Push the extension too far.** `BACK` at `+45` instead of `+40` brings the
  forearm within 1.7 mm of the rig's *back* rail — still legal, but with no
  margin for the physics arm's tracking error. `+50` fouls it outright.
- **Watch the joint errors.** `move_to` prints the worst residual after each
  waypoint. Anything above ~5° means the physics arm did not get where the plan
  assumed, and the clearance numbers no longer apply — which is why `move_to`
  raises `TrackingError` past 6° rather than carrying on.

> **If a routine is interrupted mid-flight**, the arm can end up wedged: the
> forearm threaded under the board's edge, pad below the surface, where no
> commanded pose pulls it back. `ensure_home()` will tell you so. Reset with
> `REACHY_SIM_SCENE=FWDCenterLabMCC ./scripts/start_sim.sh` and start again —
> restarting the native server resets the arm's state.

> **Both routines leave the table alone — and Routine 2 did not used to.**
> Instrumented cell by cell against a freshly reset scene, every object is still
> on its cell after the placement route *and* after the joint sweeps. That was
> not true before: `red_cube` used to start moving in §4.3 and had drifted
> 8.5 cm by §4.5, and §4.7 threw `blue_cylinder` clean off the board. The sweeps
> ran from a pose chosen when the board held nothing, and every check the
> notebook had was watching the gripper pad while the *elbow* did the damage.
> §4 now checks the whole arm along the whole path before each move; see the
> `PRESENT` comment above for the geometry, and `scene_drift()` for the evidence
> that the check is telling the truth.

## 4. Routine 2 — raise the arm, then move the gripper every way it moves

Lift out of the rest position to `PRESENT` and sweep each joint through its
range. Every move in this section is **object-aware**: before it is commanded,
the whole arm — upper arm, forearm and hand, as the three collision capsules the
MJCF actually defines — is swept along the joint-space path the move will fly,
and the move is shortened if any link would come within `SAFE_MARGIN` of
anything on the board.

That guard exists because this section used to sweep the table clean, and the
reason it did is worth reading before the first cell runs.

`PRESENT` used to be `shoulder_pitch −37.5, shoulder_roll −2, elbow −80`, and the
prose here used to say it was "about 21 cm above the board and clear of the rig,
so every angle is collision-free". The 21 cm was true, and it was about the
**gripper pad**. The pad is not the arm. At that same pose the **elbow** sat at
z = 0.778 — 3.8 cm above a 0.740 tabletop, directly over the near-right grid
cell, where `red_cube` stands 6 cm tall. The forearm was already 5 mm *inside*
the cube before any joint moved.

So the cube started drifting in §4.3, a wrist-roll sweep, where the pad never
goes anywhere near it — and by §4.7 the blue cylinder was leaving the table
entirely. Every check the notebook had was watching the pad.

Raising the pad would not have helped, because the elbow is the problem and the
elbow rides a fixed 0.28 m sphere about the shoulder. `red_cube`'s nearest
surface is **0.320 m** from the shoulder; the upper arm's own surface reaches
**0.315 m**. The near-right cell is inside the elbow's arc, and no amount of
lifting the hand changes that. The fix is to point the arm where the arc does
not sweep — up, and out to the robot's right.

| | old `PRESENT` | new `PRESENT` |
|---|---|---|
| shoulder pitch / roll / elbow | −37.5 / −2 / −80 | **−70 / −25 / −80** |
| pad above the board | 0.21 m | 0.50 m |
| elbow above the board | **0.038 m** | 0.17 m |
| worst clearance, whole arm, whole routine | **−1.9 cm** (inside `red_cube`) | **+10.9 cm** |
| …counting the table and rig too | −1.9 cm | +8.9 cm |

In [ ]:
reachy.turn_on("r_arm")
time.sleep(0.3)
prime_arm()          # safe to repeat; only matters on a fresh connection

# The margin every sweep in this section is held to.
#
# The clearance model is geometry: it knows where each link WOULD be at a
# commanded pose, not where the physics will actually leave it.  Routine 2 from
# PRESENT clears by 8.9 cm at its tightest point, so holding the guard at 5 cm
# leaves ~4 cm for the tracking error the model cannot see, and still leaves
# every intended sweep unclipped.  A clip printed below is therefore news: it
# means the geometry itself, not the margin, ran out of room.
SAFE_MARGIN = 0.05

head_on()
move_to("PRESENT", PRESENT, 3.0, tol=LESSON_TOL)
watch_hand(1.2, "the raised hand")

px, py, pz = gripper_world_xyz()
ex, ey, ez = elbow_world_xyz()
print(f"\nraised — pad   {pz - scene.table_surface_z:.3f} m above the board")
print(f"         elbow {ez - scene.table_surface_z:.3f} m above the board")
print("         (the elbow is the number that decides whether the table "
      "survives;\n          at the old PRESENT it was 0.038 m)")

print("\nwhole-arm clearance from each object at PRESENT:")
for oid, c in sorted(scene.clearances(
        link_capsules(arm_q(), gripper_deg=grip_now())).items()):
    print(f"   {oid:15s} {c.distance * 100:+6.1f} cm   (nearest link: {c.link})")
print(f"   {'table + rig':15s} "
      f"{clearance().distance * 100:+6.1f} cm   worst of everything")

### 4.1 The gripper itself — aperture

One joint, `r_gripper`, sign inverted: **negative opens**.

In [ ]:
watch_hand(0.8)
print("gripper aperture — negative opens, positive closes\n")
for label, value in [("fully open", -68.0), ("open (working default)", -45.0),
                     ("half",       -20.0), ("nearly shut",           0.0),
                     ("closed",      20.0)]:
    move_to(f"{value:+.0f}", dict(PRESENT, r_gripper=value), 0.9, report=False,
            tol=LESSON_TOL, clear=SAFE_MARGIN)
    print(f"  r_gripper = {value:+6.1f}   {label}"
          f"   (present {arm.r_gripper.present_position:+6.1f})")
    time.sleep(0.5)
move_to("open", dict(PRESENT, r_gripper=OPEN), 0.9, report=False, tol=LESSON_TOL)

### 4.2 Wrist pitch — tilt the hand up and down

`r_wrist_pitch`, ±45°. This sets the *approach angle* onto the table: positive
tips the hand up, negative down toward the surface. Its narrow range is why
Reachy 1.2 cannot do a straight top-down grasp — with a near-horizontal forearm,
45° is not enough to point the pads at the floor.

In [ ]:
watch_hand(0.8)
print("r_wrist_pitch — tilt the hand relative to the forearm (±45°)\n")
for v in (0.0, +45.0, 0.0, -45.0, 0.0):
    move_to(f"wp {v:+.0f}", dict(PRESENT, r_wrist_pitch=v), 1.8, tol=LESSON_TOL,
            clear=SAFE_MARGIN)
    time.sleep(0.4)

### 4.3 Wrist roll — rotate the hand about its own axis

`r_wrist_roll`, ±45°. Changes which way the pads face without moving the pad
*position* much. This is how you line the jaws up with an object's long axis.

In [ ]:
watch_hand(0.8)
print("r_wrist_roll — roll the hand about the forearm axis (±45°)\n")
for v in (0.0, +45.0, 0.0, -45.0, 0.0):
    move_to(f"wr {v:+.0f}", dict(PRESENT, r_wrist_roll=v), 1.8, tol=LESSON_TOL,
            clear=SAFE_MARGIN)
    time.sleep(0.4)
print()
scene_drift("after 4.3")

### 4.4 Forearm yaw — pronate and supinate

`r_forearm_yaw`, ±100°, the widest of the three. It rotates the whole forearm, so
the hand swings through a much larger arc than `r_wrist_roll` gives. Between the
two you can put the jaws at essentially any roll angle you need.

**This joint does not track under physics, and at the raised `PRESENT` it barely
tracks at all.** Measured directly at `PRESENT`, returning to `PRESENT` between
every sample so each one starts from the same place, and taken twice — after one
settle pass and after four — to separate *slow* from *stuck*:

| commanded | after 1 settle | after 4 settles |
|---|---|---|
| +15° | −24.8° | **−40.9°** |
| +45° | −11.3° | −13.0° |
| +90° | +34.9° | +68.5° |
| −45° | −9.9° | **−85.6°** |
| −90° | −59.9° | −85.2° |

Read the middle column against the right one. Extra settling makes `+90°`
better (+34.9 → +68.5) and everything else **worse**: `+15°` ends up 56° the
wrong side of its goal, and `−45°` overshoots to −85.6°. Both negative commands
converge on the same place, ≈ −85°, whatever they were asked for.

The description the data supports is that the joint slides toward ≈ −85°
whenever it is not being driven hard positive, and that more time makes that
worse rather than better. **What causes it is not established here.** The
obvious guess — gravity torque about the forearm's long axis — does not survive
the geometry: that axis is *more* vertical at the new `PRESENT` (29° from
vertical) than at the old one (63°), so the gravity moment about it is smaller,
not larger. `r_forearm_yaw` has `kp=80` against a 15 Nm limit with `kv=5`, and
the actuator model is where to look next.

> **This got worse when `PRESENT` moved.** At the old pose the pattern was
> readable — negative goals were merely slow and arrived given time, positive
> goals stuck. At the raised pose neither half holds. That is a real cost of
> the fix in §4: the new pose is what keeps the arm off the table, and it is
> worse for this joint. Both facts are measured; neither cancels the other.

> **Do not plan a grasp that depends on a precise forearm angle**, positive or
> negative, until the actuator gains are revisited.

> **The sweep below cannot show you this.** `move_to`'s printed "worst joint
> error" comes from `joint_error()`, which only polices the five `CRITICAL`
> joints — and `r_forearm_yaw` is not one of them. It is checked against
> `LOOSE_TOL` (90°), which nothing here trips. So the `fy` lines report
> `elbow_pitch` while the joint the section is about sits 50° from its goal. Read
> `arm.r_forearm_yaw.present_position` directly, as the table above does.

The sweeps in this section **report** their tracking error instead of enforcing
it — they run in open air ~47 cm above the board. What stops them reaching the
objects is not the height but the clearance guard, which checks every link along
every path; see §4's intro.

In [ ]:
watch_hand(0.8)
print("r_forearm_yaw — pronate / supinate (±100°)\n")
# Sweep to +/-90, not the joint's +/-100 hard stop.  At exactly 100 the
# actuator's ctrlrange and the joint limit are the same number, and the physics
# joint sits tens of degrees short of its goal.
#
# Return to PRESENT between samples.  Without it each reading starts from
# wherever the previous one stranded the joint, and the errors compound into
# nonsense — a sweep straight through 0 -> +90 -> 0 -> -90 -> 0 reported being
# 59.8 deg short of a commanded ZERO, which says nothing about the joint.
# The table above was measured this way, one sample per approach.
#
# Print r_forearm_yaw itself, too: move_to's "worst joint error" covers only the
# five CRITICAL joints and this is not one of them, so the convergence loop is
# blind to it as well — it will stop re-streaming once the CRITICAL joints have
# arrived, however far short this one still is.
for v in (+90.0, -90.0):
    # Unguarded: this is a RETREAT to a pose already known to be clear, and a
    # refused retreat strands the arm exactly where it should not be.
    move_to("PRESENT", PRESENT, 2.0, report=False, tol=LESSON_TOL)
    move_to(f"fy {v:+.0f}", dict(PRESENT, r_forearm_yaw=v), 3.0, report=False,
            tol=LESSON_TOL, clear=SAFE_MARGIN)
    got = arm.r_forearm_yaw.present_position
    verdict = "tracks" if abs(got - v) <= 10 else "DOES NOT CONVERGE"
    print(f"  commanded {v:+6.1f}   reached {got:+6.1f}   "
          f"short by {abs(got - v):5.1f} deg   {verdict}")
    time.sleep(0.4)

move_to("PRESENT", PRESENT, 2.0, report=False, tol=LESSON_TOL)
print(f"\n  back at PRESENT: r_forearm_yaw = "
      f"{arm.r_forearm_yaw.present_position:+.1f} (commanded 0.0)")
scene_drift("after 4.4")

### 4.5 The joints that *translate* the gripper

The three above mostly change the hand's **orientation**. To move the gripper to
a different **place** you drive the big joints. Note how much further the pad
travels per degree here.

These four sweep as **offsets from `PRESENT`**, not to fixed absolute angles.
That is a change worth naming: written absolutely, a sweep stops being "swing
this joint either way from where the arm is" and becomes "drive the arm to this
particular place", which silently breaks the moment the base pose moves. These
same four sweeps, still carrying the absolute numbers that suited the old
`PRESENT`, would fly the arm straight back down onto the board.

In [ ]:
watch_hand(0.8)
print("the joints that move the gripper's position\n")
# These four sweep RELATIVE to PRESENT, not to fixed absolute angles.
#
# They used to be absolute — arm_yaw to +/-40, shoulder_roll to -20 and +5,
# elbow to -95 and -65, shoulder_pitch to -50 and -25 — which were the right
# numbers only for the PRESENT this notebook started at.  Tie a sweep to an
# absolute angle and it stops being "swing this joint either way from here" and
# becomes "drive the arm to this specific place", which after PRESENT moved
# would have flown the arm back down onto the board on the very first sample.
# Written as offsets they follow the base pose, and the clearance guard has a
# base pose worth guarding.
for joint, deltas in [
    ("r_arm_yaw",        (0.0, -40.0, +40.0, 0.0)),
    ("r_shoulder_roll",  (0.0, -18.0,  +7.0, 0.0)),
    ("r_elbow_pitch",    (0.0, -15.0, +15.0, 0.0)),
    ("r_shoulder_pitch", (0.0, -12.5, +12.5, 0.0)),
]:
    print(f"  {joint}  (about {PRESENT[joint]:+.1f}):")
    for d in deltas:
        move_to(f"{d:+.0f}", dict(PRESENT, **{joint: PRESENT[joint] + d}), 1.4,
                tol=LESSON_TOL, clear=SAFE_MARGIN)
        time.sleep(0.3)
print()
scene_drift("after 4.5")

### 4.6 All together — a wave

Multi-joint targets interpolate simultaneously, which is what makes motion look
deliberate rather than sequential.

In [ ]:
WAVE_A = dict(PRESENT, r_forearm_yaw=-60.0, r_wrist_pitch=25.0, r_wrist_roll=-30.0)
WAVE_B = dict(PRESENT, r_forearm_yaw=+60.0, r_wrist_pitch=-25.0, r_wrist_roll=+30.0)

watch_hand(0.8)
print("combined motion — watch RViz\n")
for i in range(3):
    # Three weak joints reversing together — 0.9 s per swing left them tens of
    # degrees short and tripped the tracking guard.
    move_to(f"wave {i+1}a", WAVE_A, 1.8, report=False, tol=LESSON_TOL,
            clear=SAFE_MARGIN)
    move_to(f"wave {i+1}b", WAVE_B, 1.8, report=False, tol=LESSON_TOL,
            clear=SAFE_MARGIN)
    print(f"  wave {i + 1}/3")
move_to("PRESENT", PRESENT, 1.8, tol=LESSON_TOL)   # retreat: unguarded
scene_drift("after 4.6")

### 4.7 Pointing at the grid — from joint angles to world coordinates

`CartesianPlanner` wraps the SDK's IK so you can ask for a **world position**
instead of joint angles. It plans in *pad* space (the contact point between the
jaws, ~0.12 m along the wrist's local −Z).

Two things make this work at all on an occupied board, and both were measured
rather than guessed.

**The IK spends the arm's redundancy on the table now.** The arm is 7-DOF, so
many orientations put the pad on the same point with the elbow somewhere
completely different. The default rule picks among them by pad error — a
criterion that knows nothing about what is standing on the board — and stops at
the first solution within a millimetre. `maximise_clearance=True` scores every
candidate by whole-arm clearance instead:

| target | default rule | roomiest |
|---|---|---|
| `cell_r2c2` @ 28 cm | +0.6 cm | **+12.1 cm** |
| `cell_r2c3` @ 28 cm | +0.8 cm | **+13.4 cm** |
| `cell_r3c3` @ 18 cm | +1.9 cm | **+13.4 cm** |

**The hand travels shut.** The gripper's moving finger swings outward as it
opens — 5.2 cm about the wrist axis shut, 7.5 cm at the working open angle,
8.2 cm wide. `blue_cylinder` stands in the middle of the grid so every reach
crosses over it, and with an open hand that left 0.6–2.0 cm and refused nearly
every cell. This is what `PLACE_ROUTE` has always done through the rig, and the
numbers now say why: at `SWING_1` a shut hand clears the outer rail by 6 mm and
an open one fouls it by 16 mm.

**The margin is 5 cm, from the right measurement.** It was briefly 20 cm, taken
from the worst *pad* miss (21.9 cm), which shut the whole section down. The pad
is not what approaches anything. Measured properly — plan a cell, fly it, then
recompute clearance from the joints the arm actually reached — the guard's
answer degrades by at most **0.9 cm**, across moves carrying up to 47° of joint
error. The joints that carry that error are `forearm_yaw` and the wrists, which
only spin the hand about its own axis; the links that approach objects are
placed by shoulder, arm_yaw and elbow, which track to a few degrees.

**Every approach is flown in six legs, and the arm is read back between them.**
This is the part that stopped the section knocking things over, and it is worth
being precise about why, because four earlier fixes here were plausible and
insufficient.

The plan-time guard describes an arm that tracks perfectly. Calibrated at the
*endpoints* of each move — plan a cell, fly it, recompute clearance from the
joints actually reached — it looked trustworthy: the answer degraded by at most
**0.9 cm**. Then `cell_r2c1` reported **+5.5 cm** to the soda can and moved it
**0.189 m**. Both measurements are correct. The 0.9 cm was taken at the ends of
the move; the can was hit in the middle, where nothing was measured at all.
`goto` interpolates in joint space and every joint runs monotonically to its
goal, but not *in step* — the shoulder (kp 300) arrives well before the wrists
(kp 60), so the arm bows out of the corridor the guard cleared. No margin fixes
that, because nothing bounds the excursion in advance.

`escort_to` cuts the move into legs. Before each leg the same check runs, but
**from the pose the arm is actually in**; after each leg the arm is read back
and the clearance that really happened is measured. Under `margin / 2` it stops
and retraces to the last leg that measured clean.

Measured on this scene, one full clean run:

| | one-shot check | in legs | + per-object margins |
|---|---|---|---|
| cells flown | 4 / 9 | 6 / 9 | **7 / 9** |
| objects hovered | 1 / 4 | 2 / 4 | **4 / 4** |
| `cell_r2c1` | flown on a claimed +5.5 cm, can moved 0.189 m | stopped at 67% | **flown, board untouched** |
| `blue_cylinder` | — | moved 0.123 m, unremarked | **held** |
| whole section | board disturbed | disturbed on the object pass | **every object still on its cell** |
| worst plan-vs-real gap | ≤0.9 cm (endpoints) | +2.8 cm | **+6.2 cm** |

That last row is the whole argument in one number, and the `degraded` column in
the output is where to watch it. It is planned-minus-realised on the worst leg.
Near zero means the plan-time guard was telling the truth and the legs only cost
time; when it jumps, the guard was about to be wrong.

Refusals are the guard working, not failing, and the sharpest one so far reads:

    cell_r1c3  STOPPED — hand OVERLAPS red_cube by 1.7 cm once leg 6 was flown
                         (planned hand clears red_cube by 6.0 cm)

A **7.7 cm** swing between what the plan promised and what the flight did. In
another run the same kind of stop named a different object than the plan was
watching — 2.5 cm from the soda can while the plan tracked `red_cube` at 5.5 cm.
The plan is not merely optimistic about a distance; it can be watching the wrong
object entirely, and no margin fixes that.

One honest caveat on that stop: the drift check right after it read *every
object still on its cell*, so a modelled 1.7 cm overlap moved the cube less than
5 mm. `hand_radius` returns the worst-case thumb arc, so the hand capsule
overstates the real envelope and some refusals are false alarms. That is the
right direction to err, and it is why refusals cost coverage rather than safety.

**Expect 6–8 of 9 cells.** `cell_r3c1` is unreachable at every height — the
arm's own limit, the same one Siva confirmed on the physical robot, not a
refusal.

**The section then hovers over each object**, which is what a grasp actually has
to do — pad above the object's *top*, so a tall can and a flat block get the
same air underneath.

> **The object being approached is now checked too**, which it was not until
> recently. `approaching=oid` used to drop the target from the obstacle set —
> necessary-looking, because hovering 6 cm over a can puts the *hand*, a 5.2 cm
> capsule whose tip is the pad, about 1 cm from it, and one global 5 cm margin
> refuses every approach that could ever be made. But dropping it meant nothing
> guarded the one object the arm was aimed at: `blue_cylinder` was hovered to
> 1.6 cm with 6.1 cm reported, moved 0.123 m, and `degraded` read +0.6 cm.
>
> It now keeps its own margin, derived from the pose being flown to: whatever
> the destination hover achieves, less `APPROACH_SLACK`. You can see it working
> in the output — `red_cube` hovers with `clearance +0.5 cm (red_cube)`, the
> target itself binding, honestly reported and allowed. All four objects hover
> and the board is untouched.
>
> The pass **stops at the first disturbed object** rather than ploughing on.

In [ ]:
# Hover height, derived rather than assumed.  This number is about POINTING;
# it is not what keeps the objects on the table — the clearance guard is.
#
# The old value added a Z_SLOP term of 0.10 m on top of the tallest object, on
# the reasoning that the arm does not fly the height it is given.  That bought
# nothing: a run with the taller hover still threw the blue cylinder 1.06 m off
# the board, because the object was hit by the FOREARM in transit and no pad
# height addresses that.  What the extra 10 cm did do was wreck the pointing it
# exists to demonstrate.
CLEARANCE = 0.06        # air we want under the pad

_tops = [scene.get(o).top_z for o in scene.manipulable_ids()]
_tallest = max(_tops) if _tops else scene.table_surface_z
HOVER_H = max(0.12, (_tallest - scene.table_surface_z) + CLEARANCE)

# THE MARGIN, AND WHY IT IS 5 cm AND NOT THE 20 cm IT USED TO BE.
#
# The guard is a plan-time check: it measures the arm at the pose about to be
# commanded, which is only worth something if the arm then flies roughly that
# pose.  The first attempt at this number came from the worst measured PAD miss
# over this grid — 21.9 cm — and set the margin above it.  That was the wrong
# quantity, and it shut the whole section down.
#
# The right question is not "how far is the pad from target" but "how much does
# the GUARD'S ANSWER change between the commanded pose and the realised one",
# because the pad is not the part that approaches anything.  Measured directly,
# planning each cell, flying it, then recomputing clearance from the joint
# angles the arm actually reached:
#
#     cell     planned  realised   degraded   joint err   pad miss
#     r1c1      +12.4     +12.4       0.0       47.2 deg    9.6 cm
#     r1c2      +12.1     +12.1       0.0       47.6 deg   21.2 cm
#     r1c3      +12.9     +12.9       0.0       25.6 deg    9.1 cm
#     r2c1      +10.6      +9.6      +0.9       11.2 deg    2.0 cm
#     r2c2      +12.1     +12.3      -0.3       34.1 deg    5.3 cm
#     r2c3      +12.1     +11.7      +0.4       37.3 deg    4.2 cm
#
# The guard degrades by at most 0.9 cm across moves carrying up to 47 deg of
# joint error and 21 cm of pad miss.  That looks paradoxical until you notice
# which joints carry the error: forearm_yaw and the wrists, which only spin the
# hand about its own axis.  The upper arm and forearm — the links that actually
# come near objects — are placed by shoulder pitch/roll, arm_yaw and elbow,
# and those track to a few degrees.  The pad accumulates every joint's error at
# the end of a half-metre lever; the elbow accumulates three joints' worth over
# 28 cm.
#
# So 5 cm is roughly five times the measured degradation, and the same value the
# sweeps use.  Every cell above was flown at it with the board untouched.
#
# AND THEN cell_r2c1 REPORTED +5.5 cm AND MOVED THE CAN 0.189 m.  Both numbers
# above are still true; they are just answers to a question that was too narrow.
# The 0.9 cm was measured at the ENDPOINTS of each move.  The can was hit in the
# MIDDLE of one, where nothing was measured at all — `goto` interpolates in
# joint space, but the joints do not arrive in step, so the arm bows off the
# line the guard cleared.  That is not something a bigger margin fixes, because
# nothing bounds the excursion in advance.  It is fixed by flying the move in
# pieces and looking at the arm between them: see `escort_to`.
POINT_MARGIN = SAFE_MARGIN

# How many hops each approach is broken into.  This is the knob that decides how
# good an approximation `path_clearance` is: it models the arm between two poses
# as a joint-space straight line, which is exactly the assumption that failed
# above — but over a sixth of a move the joints have far less room to get out of
# step, so it is much closer to true.  More legs is a better model and a slower,
# more stuttery demo; six keeps a 2 s approach under 3 s and puts a measurement
# every ~4 cm of pad travel.  It never reaches zero error, and pretending
# otherwise is how the last four versions of this guard were wrong.
POINT_LEGS = 6

# Where the base hover is not enough, the target is LIFTED rather than skipped.
#
# Tried once before and abandoned, because it made things worse: it lifted until
# each cell passed a 5 cm check, flew them all with a clean guard, and the drift
# check afterwards read "blue_cylinder moved 5.830 m".  Both reasons for that
# are now gone — the solver spends the arm's redundancy on clearance instead of
# throwing it away, and the margin is derived from the right measurement.  With
# those fixed the lift is sound, and it is needed: at the base hover only two of
# the nine cells clear, because 17 cm over the table is barely 6 cm over a can.
#
# The search steps upward and takes the first height that clears rather than
# bisecting: clearance is NOT monotonic in hover height, since the IK returns a
# different arm configuration at each target.
LIFT_STEP = 0.05
LIFT_MAX = 0.25

# How much worse than the planned hover the approach may get before it is
# stopped.  A tolerance on a DERIVED number, not a clearance in its own right:
# the destination hover already sits about 1 cm off the target, so there is no
# room to demand centimetres.  2 cm below that is comfortably inside the object,
# which is the thing worth refusing.
APPROACH_SLACK = 0.02

print(f"base hover {HOVER_H * 100:.0f} cm, lifting in {LIFT_STEP * 100:.0f} cm "
      f"steps to +{LIFT_MAX * 100:.0f} cm; whole-arm margin "
      f"{POINT_MARGIN * 100:.0f} cm")
print("(the IK sweeps up to 54 orientations per target — give it a few seconds)")
print(f"every approach is flown in {POINT_LEGS} legs, with the arm read back and "
      f"re-measured\nafter each one; it stops if the measured clearance falls "
      f"under {POINT_MARGIN * 50:.1f} cm\n")

watch_table(1.2, "the board")
scene_drift("before")
print()


def point_at(label, xy, base_z, secs=2.0, approaching=None):
    """Hover the pad over one spot, lifting until the whole arm has room.

    `maximise_clearance` is what makes this possible at all.  The arm is
    redundant — many orientations put the pad on the same point with the elbow
    somewhere completely different — and the default IK rule spends that
    redundancy on the last millimetre of pad error, a criterion that knows
    nothing about the table.  Scoring candidates by whole-arm clearance instead
    turned cell_r2c2 from +0.6 cm to +12.1 cm at the same target.

    `approaching` names the object being reached for.  It gets a margin of its
    own rather than being dropped from the check, and the difference between
    those two is a defect that took a live run to see.

    The reason it needs special treatment is real: hovering 6 cm over the can
    puts the HAND — a 5.2 cm capsule whose tip is the pad — about 1 cm from the
    can, so holding it to the same 5 cm as everything else refuses every
    approach that could ever be made.  The first fix was to exclude it, which
    meant NOTHING guarded the object being reached for.  Measured:
    blue_cylinder hovered to within 1.6 cm with 6.1 cm of clearance reported,
    moved 0.123 m, and `degraded` read +0.6 cm — the loop reported a clean
    flight because it was not looking at the one object the arm was aimed at.

    Now the target stays in the check with its margin DERIVED from the pose
    being flown to: whatever clearance the destination hover actually achieves,
    less APPROACH_SLACK.  Self-consistent by construction — the approach may be
    exactly as close as it has to be and no closer — while a flight that walks
    through the object still trips, because the far side of it is not on that
    path.  The SOLVER still ignores the target: scoring candidate poses by
    distance from the thing you are trying to reach would push the IK away from
    it.  Optimise the approach, then verify it.

    THE HAND TRAVELS SHUT.  Not tidiness — the gripper's moving finger swings
    out as it opens, so an open hand is a 7.5 cm tube about the wrist axis and
    a shut one is 5.2 cm.  blue_cylinder stands in the middle of the grid and
    every reach across the board passes over it, which with an open hand left
    0.6-1.9 cm and refused nearly every cell.  Closing the hand for transit is
    what the placement route already does through the rig, and for the same
    reason: at SWING_1 the shut hand clears the outer rail by 6 mm and an open
    one fouls it by 16 mm.  Open the hand when you arrive, not before.
    """
    # What the SOLVER optimises against: everything except the target, because
    # scoring candidate poses by distance from the object you are reaching for
    # drives the IK away from it.  The GUARD below sees everything.
    solve_ids = None
    if approaching is not None:
        solve_ids = [o for o in scene.obstacle_ids(include_static=True)
                     if o != approaching]
    # Start every target from PRESENT, not from wherever the last one left the
    # arm.  Same reason section 4.4 returns between samples: planning from a
    # stranded pose compounds.  Observed without it — one awkward parking spot
    # after cell_r2c1 and the next five targets all refused with the identical
    # "hand clears soda_can by 3.4 cm", which is the arm's position talking, not
    # the board's.  The retreat is unguarded on purpose (see move_to).
    move_to("PRESENT", PRESENT, 2.0, report=False, tol=LESSON_TOL)
    lift = 0.0
    best = None
    reason = None
    while lift <= LIFT_MAX + 1e-9:
        refresh_scene()
        seed = arm_q()
        target = (xy[0], xy[1], base_z + lift)
        try:
            sol = planner.solve(target, seed=seed, maximise_clearance=True,
                                from_joints=seed, gripper_deg=SHUT,
                                ids=solve_ids, include_static=True)
        except UnreachableError as exc:
            if best is None:
                reason = f"UNREACHABLE — {exc}"
            lift += LIFT_STEP
            continue
        # The target's margin comes from the pose being flown TO: measure what
        # the destination hover gives against that object, then require the way
        # in not to fall APPROACH_SLACK below it.  Re-derived every attempt,
        # because a lifted target is a different pose with a different answer.
        margins = {}
        if approaching is not None:
            rest = planner.clearances(sol, SHUT, ids=[approaching],
                                      include_static=True).get(approaching)
            if rest is not None:
                margins[approaching] = max(0.0, rest.distance - APPROACH_SLACK)
        c, need = clearance_binding(
            planner.path_clearances(seed, sol, gripper_deg=SHUT,
                                    include_static=True),
            POINT_MARGIN, margins)
        # Report the BEST attempt, not the last one tried — a refusal that
        # quotes whichever lift happened to be final tells you nothing about
        # how close the target came to being reachable.
        if best is None or c.distance > best.distance:
            best = c
            reason = f"{c} on the way in (best of {lift * 100:.0f} cm lift)"
        if c.distance >= need:
            flight = escort_to(label, dict(zip(R_ARM_JOINTS, sol),
                                           r_gripper=SHUT),
                               secs, margin=POINT_MARGIN, margins=margins,
                               legs=POINT_LEGS, gripper_deg=SHUT,
                               include_static=True, report=False)
            if not flight.completed:
                # NOT a planning failure.  The plan was checked and it passed —
                # this is the flight disagreeing with it, which is precisely the
                # information the one-shot check never had.  Do not retry at a
                # higher lift: the arm has just retreated from something, the
                # board may no longer be where the model thinks, and a second
                # approach to the same spot is the move most likely to finish
                # what the first one started.
                print(f"  {label:11s} STOPPED at {flight.fraction * 100:3.0f}% "
                      f"of leg {len(flight.legs)}/{POINT_LEGS} — "
                      f"{flight.reason}")
                return None
            px, py, pz = gripper_world_xyz()
            miss = sum((a - b) ** 2 for a, b in zip((px, py, pz), target)) ** 0.5
            note = f"   lifted +{lift * 100:.0f} cm" if lift else ""
            # `degraded` is the number to watch across runs: planned minus
            # realised, worst leg.  Near zero means the plan-time guard was
            # telling the truth and the legs only cost time.  When it jumps, the
            # guard was about to be wrong and the loop is what noticed.
            print(f"  {label:11s} pad ({px:+.3f}, {py:+.3f}, {pz:.3f})   "
                  f"miss {miss * 100:5.1f} cm   clearance "
                  f"{c.distance * 100:+5.1f} cm ({c.object_id})   degraded "
                  f"{flight.worst_degraded * 100:+4.1f} cm{note}")
            return miss
        reason = f"{c} on the way in"
        lift += LIFT_STEP
    print(f"  {label:11s} NOT FLOWN — {reason}")
    return None


print("hovering over each grid cell:")
cell_miss, refused = [], []
for cid in scene.grid_cells():
    cx, cy, cz = scene.cell_center(cid)
    m = point_at(cid, (cx, cy), cz + HOVER_H)
    (cell_miss if m is not None else refused).append(m if m is not None else cid)
    # Per cell, not once at the end.  Checking only at the end of the pass told
    # us foam_block had moved but not which cell moved it, which is the one
    # thing you need to debug it.
    #
    # AND AFTER A REFUSAL, not only after a flight.  This used to be guarded by
    # `m is not None`, which skipped the check at the exact moment it is worth
    # most: the closed-loop guard has just said the arm ended a leg 2.5 cm from
    # something, and "did anything actually move" is the only question left.
    # Measured cost of that omission — a run where `cell_r2c1` stopped at 83%,
    # no drift line was printed, and the NEXT cell reported the can 4.341 m off
    # the board.  Two candidates, no way to tell which, because the one check
    # that would have separated them was skipped for being on a refusal.
    if scene_drift(cid):
        print("   stopping the grid pass: the board is no longer where the "
              "guard thinks it is")
        break
    time.sleep(0.3)

# ── Hovering over the objects themselves ─────────────────────────────────────
#
# This is the thing a grasp actually has to do: put the pad over the object's
# TOP rather than over the table, so a tall can and a flat block get the same
# air under the pad.
#
# READ THE DRIFT LINES ON THIS PASS.  It is the least reliable part of the
# notebook and it is on by choice, not because it is finished.  Measured: the
# guard reported 6-8 cm of clearance on every move it allowed and a run still
# put blue_cylinder 1.19 m off the board.
#
# Four separate causes have been found and fixed in this guard already — it
# checked only the pad, then only the endpoints, then against stale object
# positions, then with a hand capsule sized for a closed gripper.  Each fix was
# real; each time a run surfaced another gap.  The pattern is the useful part: a
# PLAN-TIME check cannot cover close-quarters work on this arm.  It verifies the
# pose it is about to command, and between that and what the physics does there
# is enough slack to reach something 6 cm away.  The next step is closing the
# loop — step the move, re-check the pose actually reached, abort when the
# realised clearance drops — not another constant.
#
# What this pass DOES do is stop at the first sign of trouble.  Left running
# after a knock it kept ploughing and took blue_cylinder from 0.88 m to 1.19 m
# across successive moves; now one dirty drift check ends the pass.
HOVER_OVER_OBJECTS = True

print("\nhovering over each object:")
obj_miss = []
for oid in (scene.manipulable_ids() if HOVER_OVER_OBJECTS else []):
    watch_table(0.6)
    hx, hy, hz = scene.hover_point(oid, CLEARANCE)
    m = point_at(oid, (hx, hy), hz, secs=2.2, approaching=oid)
    if m is not None:
        obj_miss.append(m)
    if scene_drift(oid):
        print("   stopping the object pass: the board has been disturbed, and "
              "every further check would be built on a wrong model")
        break
    time.sleep(0.3)
if not HOVER_OVER_OBJECTS:
    print("  skipped — HOVER_OVER_OBJECTS is False")

if cell_miss:
    print(f"\ncells flown {len(cell_miss)}/{len(scene.grid_cells())}   "
          f"worst miss {max(cell_miss) * 100:.1f} cm   "
          f"(grid pitch is 15.2 cm — a miss above that is the wrong cell)")
if refused:
    print(f"not flown: {', '.join(refused)}")
if obj_miss:
    print(f"objects hovered {len(obj_miss)}/{len(scene.manipulable_ids())}   "
          f"worst miss {max(obj_miss) * 100:.1f} cm")

# The guard is a model, and a model can be wrong.  This is not.  Read the two
# together: a clean guard and a clean drift check mean the geometry and the
# physics agree; a clean guard and a dirty drift check means the model is
# missing something and the model is what needs fixing.
print()
scene_drift("after")

# Retreat — deliberately unguarded.  A refused retreat strands the arm exactly
# where it should not be; see move_to's docstring.
move_to("PRESENT", PRESENT, 2.0, tol=LESSON_TOL)
watch_hand(1.0, "back to the hand")

## 5. Stow — reverse the route back into the pocket

Retrace the placement route backwards. Do not shortcut it: going straight from
`PRESENT` to `HOME` drives the upper arm through the rig's front rail.

In [ ]:
watch_hand(0.8)
print("stowing — back into the pocket\n")
for name, target, secs, tol in STOW_ROUTE:
    move_to(name, target, secs, tol=tol)
    time.sleep(0.3)

scene_drift("after stow")
look_at((0.45, 0.0, 1.05), 1.0, "level, at rest")
reachy.turn_off("r_arm")
reachy.turn_off("head")
print(f"\narm at rest in the rail pocket (at HOME: {at_pose(HOME, joints=GROSS_JOINTS)}), motors off.")

## 6. Practice

1. **Re-time the placement.** Halve every duration in `PLACE_ROUTE` and watch the
   residual joint errors `move_to` prints grow. Above ~5° the verified clearances
   stop applying.
2. **Rest the hand elsewhere on the board.** `REST` puts the pad at
   (0.539, −0.249). Shift `r_shoulder_roll`, check with `gripper_world_xyz()`,
   and keep the whole hand inside y ∈ [−0.349, +0.349].
3. **Touch a cell instead of hovering.** Drop `HOVER_H` to 0.02 and see which
   cells still solve.
4. **Write your own routine** as `(name, pose, duration)` tuples, and verify it
   offline before running — the MuJoCo contact check used to build this notebook
   is in `native_mujoco/`.
5. **Promote what works** into `src/reachy_ai/motion/primitives.py`. That is the
   only place motion may come from on the physical robot.

### The open questions this notebook exposes

Two of the three rig numbers `scenes/FWDCenterLabMCC.yaml` used to carry as
*assumed* have been settled from the photographs and the operator, and both were
wrong in the same direction — they put aluminium where there is none:

- **The rails' height.** The board **rests on top of** the frame
  (`docs/pics/80903696209` shows its laminate edge proud of the rail beneath it).
  The rails had been modelled flush with the board's *surface*, putting 25 mm of
  phantom aluminium in the plane the arm has to cross.
- **The front cross rail.** There isn't one. That edge is finished with wooden
  trim overhanging open air (`docs/pics/80903693586`). A `rig_rail_front` had
  been modelled squarely across the arm's route — the single largest obstruction
  in the scene.

Correcting both transformed the swing: its tightest points went from 1.4 mm and
0.3 mm to 23 mm and 4.8 mm.

**One number is still open, and it is now the binding constraint** — the board's
**robot-side edge**, at x = 0.160 in the scene and never measured. The elbow
crosses it at z = 0.770 carrying a 35 mm collision radius, which is the 4.8 mm in
the clearance table above. At x = 0.190 it would clear by 20 mm instead.

Removing the phantom front rail also removed this edge's lower bound, so it is no
longer even bracketed. A related loose end: with no front member the pocket
measures ~20.8 in fore-aft rather than the 19 in the setup notes record — so
either a front member sits further forward than the photos show, or that edge is
closer to the robot than 0.160. One tape measure from the pedestal axis to the
board's near edge settles both.